# Building Capacity Analysis

This notebook provides a comprehensive analysis of building capacities based on geographical factors.

In [1]:
import pandas as pd

from analysis.building_levels.building_analysis import CapacityAnalyzer, get_path

analyzer = CapacityAnalyzer()
print("Analyzer initialized.")

Loading live game data...
Analyzer initialized.


## Modifier Sources Breakdown

In [2]:
for building in analyzer.building_map.keys():
    df = analyzer.get_modifier_sources_df(building_name=building)
    # Keep only non-total rows with non-zero values for this building
    df_filtered = df[~df["Source"].str.contains("TOTAL|ALL SOURCES", case=False, na=False)]
    df_filtered = df_filtered[df_filtered[building].notna() & (df_filtered[building] != 0)]
    if not df_filtered.empty:
        display_section = df_filtered[["Category", "Source", building]].rename(columns={building: "Bonus"})
        print(f"### {building}")
        display(display_section)
        s = analyzer.scaling[building]
        base = analyzer.base_levels[building]
        print(f"  Also: base={base}, +{s['dev']} per development, +{s['pop']} per population")

### Fruit Orchard


,Category,Source,Bonus
0,Climate,mediterranean,3.0
1,Climate,subtropical,2.0
2,Climate,oceanic,1.0
3,Climate,tropical,1.0
4,Climate,cold_arid,-2.0
5,Climate,arctic,-3.0
8,Topography,hills,2.0
9,Topography,mountains,-2.0
10,Topography,wetlands,-3.0
13,Vegetation,farmland,1.0


  Also: base=0.0, +0.15 per development, +0.025 per population
### Sheep Farm


,Category,Source,Bonus
0,Climate,mediterranean,1.0
1,Climate,arid,2.0
2,Climate,oceanic,1.0
3,Climate,continental,1.0
4,Climate,tropical,-1.0
5,Climate,cold_arid,2.0
8,Topography,hills,3.0
9,Topography,flatland,1.0
10,Topography,plateau,2.0
11,Topography,mountains,2.0


  Also: base=0.0, +0.15 per development, +0.025 per population
### Farming Village


...

  Also: base=0.0, +0.15 per development, +0.025 per population
### Fishing Village


,Category,Source,Bonus
0,Climate,mediterranean,1.0
1,Climate,subtropical,1.0
2,Climate,oceanic,1.0
3,Climate,tropical,2.0
4,Climate,arctic,1.0
7,Topography,hills,-2.0
8,Topography,plateau,-2.0
9,Topography,mountains,-3.0
10,Topography,wetlands,1.0
11,Topography,salt_pans,-2.0


  Also: base=0.0, +0.15 per development, +0.025 per population
### Forest Village


,Category,Source,Bonus
0,Climate,arid,-1.0
1,Climate,continental,1.0
2,Climate,arctic,1.0
5,Topography,hills,1.0
6,Topography,mountains,1.0
9,Vegetation,woods,2.0
10,Vegetation,desert,-1.0
11,Vegetation,forest,3.0
12,Vegetation,jungle,1.0
15,Rank,city,-6.0


  Also: base=0.0, +0.15 per development, +0.025 per population


In [3]:
# 19.24 * 0.15 2.886
# 24.76 * 0.025 0.619
# oceanic 1
# coastal 5
# fish 8

# Category	Source	Bonus
# 0	Climate	mediterranean	2.0
# 1	Climate	subtropical	1.0
# 2	Climate	oceanic	1.0
# 3	Climate	tropical	2.0
# 4	Climate	arctic	1.0
# 7	Topography	hills	-2.0
# 8	Topography	plateau	-2.0
# 9	Topography	mountains	-3.0
# 10	Topography	wetlands	1.0
# 11	Topography	salt_pans	-2.0
# 12	Topography	atoll	5.0
# 15	Vegetation	desert	-2.0
# 16	Vegetation	jungle	1.0
# 19	Static	coastal	5.0
# 20	Static	river	2.0
# 21	Static	lake	3.0
# 24	RGO	fish	8.0


## Generalized Analysis Interface

The `CapacityAnalyzer` now supports generalized filtering and analysis methods. This allows for quick custom queries without writing new scripts.

In [4]:
df_locations = analyzer.location_data.get_merged_df()
df_locations[["location", "is_ownable"]].value_counts("is_ownable")


# df_locations.loc[df_locations.is_ownable == False].sample(10)[["location", "topography"]]
# df_locations.loc[(df_locations.is_ownable == False)][["location", "topography"]].value_counts("topography")
# df_locations.loc[(df_locations.is_ownable == True)][["location", "topography"]].value_counts("topography")
# df_locations.loc[(df_locations.is_ownable == True) & (df_locations.topography == "mesa_wasteland")][["location", "topography"]]
# df_locations.loc[(df_locations.is_ownable == False) & (df_locations.topography == "hills")][["location", "topography"]].value_counts("topography")
# df_locations.loc[(df_locations.is_ownable == False) & (df_locations.topography == "hills")][["location", "topography"]]

is_ownable
True     21214
False     7359
Name: count, dtype: int64

### Query by Country TAG

Single aggregate sheet: population, development, building capacity (and relative share), employment and employment/pop ratio per building, geography breakdown (climate/topography/vegetation %).

In [5]:
TAG = "ENG"  # Change to query any country (e.g. FRA, ENG, POL)

country_locs = df_locations[df_locations["owner_tag"] == TAG].copy()
country_locs = country_locs.dropna(subset=["climate", "topography", "vegetation"])
if "rank" not in country_locs.columns:
    country_locs["rank"] = "rural_settlement"

n_loc = len(country_locs)
if n_loc == 0:
    print("No owned locations found. Check TAG.")
else:
    total_pop = country_locs["population"].sum()
    avg_dev = country_locs["development"].mean().round(2)
    df_cap = analyzer.calculate_capacities_for_locations(country_locs)
    building_cols = ["Fruit Orchard", "Sheep Farm", "Farming Village", "Fishing Village", "Forest Village"]
    cap_sum = df_cap.groupby("Building")["Total Bonus"].sum().reindex(building_cols)
    total_cap = cap_sum.sum()
    bd = analyzer.building_data.modded_df
    building_keys = {"Fruit Orchard": "fruit_orchard", "Sheep Farm": "sheep_farms", "Farming Village": "farming_village", "Fishing Village": "fishing_village", "Forest Village": "forest_village"}
    emp_vals = [bd.loc[building_keys[b], "employment_size_val"] if building_keys[b] in bd.index else 0 for b in building_cols]
    employment = cap_sum * pd.Series(emp_vals, index=building_cols)
    emp_ratio = (employment / total_pop * 100).round(2) if total_pop > 0 else pd.Series(float("nan"), index=building_cols)

    # Build single aggregate row
    row = {
        "TAG": TAG,
        "n_locations": n_loc,
        "total_population_k": round(total_pop, 1),
        "avg_development": avg_dev,
    }
    for b in building_cols:
        c, e, r = cap_sum.get(b, 0), employment.get(b, 0), emp_ratio.get(b, float("nan"))
        pct = (c / total_cap * 100) if total_cap > 0 else 0
        short = b.replace(" ", "_").lower()
        row[f"{short}_cap"] = round(c, 1)
        row[f"{short}_pct"] = round(pct, 1)
        row[f"{short}_emp_k"] = round(e, 1)
        row[f"{short}_emp_pop_pct"] = r if pd.notna(r) else "-"
    for col in ["climate", "topography", "vegetation"]:
        if col not in country_locs.columns:
            continue
        vc = country_locs[col].value_counts(normalize=True) * 100
        for val, pct in vc.items():
            row[f"{col}_{val}_pct"] = round(pct, 1)

    # Order columns: core, buildings (cap/pct/emp/emp_pop), geography
    core = ["TAG", "n_locations", "total_population_k", "avg_development"]
    building_cols_short = ["fruit_orchard", "sheep_farm", "farming_village", "fishing_village", "forest_village"]
    building_cols_ordered = [f"{b}_{s}" for b in building_cols_short for s in ["cap", "pct", "emp_k", "emp_pop_pct"]]
    geo_cols = [c for c in row if c not in core and c not in building_cols_ordered]
    df_country = pd.DataFrame([row])[[c for c in core + building_cols_ordered + geo_cols if c in row]]
    display(df_country)

    # Compact view: fewer columns, stacked values, padded with empty where lengths differ
    summary_list = [TAG, f"{n_loc} loc", f"{total_pop:.0f}k pop", f"avg dev {avg_dev}"]
    veg_list = [f"{v}: {pct:.1f}%" for v, pct in country_locs["vegetation"].value_counts(normalize=True).mul(100).items()]
    topo_list = [f"{v}: {pct:.1f}%" for v, pct in country_locs["topography"].value_counts(normalize=True).mul(100).items()]
    climate_list = [f"{v}: {pct:.1f}%" for v, pct in country_locs["climate"].value_counts(normalize=True).mul(100).items()]
    cap_list = [f"{b}: {cap_sum.get(b, 0):.1f}" for b in building_cols]
    pct_list = [f"{b}: {(cap_sum.get(b,0)/total_cap*100):.1f}%" if total_cap > 0 else f"{b}: -" for b in building_cols]
    emp_list = [f"{b}: {employment.get(b, 0):.1f}k" for b in building_cols]
    emp_pop_list = [f"{b}: {emp_ratio.get(b):.1f}%" if pd.notna(emp_ratio.get(b)) else f"{b}: -" for b in building_cols]
    n_rows = max(1, len(veg_list), len(topo_list), len(climate_list), len(cap_list))
    def pad(lst, n):
        return lst + [""] * (n - len(lst))
    df_compact = pd.DataFrame({
        "Summary": pad(summary_list, n_rows),
        "Vegetation": pad(veg_list, n_rows),
        "Topography": pad(topo_list, n_rows),
        "Climate": pad(climate_list, n_rows),
        "Capacity": pad(cap_list, n_rows),
        "Relative Cap %": pad(pct_list, n_rows),
        "Employment": pad(emp_list, n_rows),
        "Emp/Pop %": pad(emp_pop_list, n_rows),
    })
    print("\nCompact view:")
    display(df_compact)

,TAG,n_locations,total_population_k,avg_development,fruit_orchard_cap,fruit_orchard_pct,fruit_orchard_emp_k,fruit_orchard_emp_pop_pct,sheep_farm_cap,sheep_farm_pct,...,forest_village_emp_pop_pct,climate_oceanic_pct,topography_flatland_pct,topography_wetlands_pct,topography_hills_pct,vegetation_grasslands_pct,vegetation_woods_pct,vegetation_sparse_pct,vegetation_farmland_pct,vegetation_forest_pct
0,ENG,138,3015.5,20.37,696.6,16.8,3483.2,115.51,1038.5,25.1,...,57.43,100.0,87.7,6.5,5.8,42.8,22.5,12.3,11.6,10.9



Compact view:


,Summary,Vegetation,Topography,Climate,Capacity,Relative Cap %,Employment,Emp/Pop %
0,ENG,grasslands: 42.8%,flatland: 87.7%,oceanic: 100.0%,Fruit Orchard: 696.6,Fruit Orchard: 16.8%,Fruit Orchard: 3483.2k,Fruit Orchard: 115.5%
1,138 loc,woods: 22.5%,wetlands: 6.5%,,Sheep Farm: 1038.5,Sheep Farm: 25.1%,Sheep Farm: 5192.6k,Sheep Farm: 172.2%
2,3016k pop,sparse: 12.3%,hills: 5.8%,,Farming Village: 1238.0,Farming Village: 29.9%,Farming Village: 6190.1k,Farming Village: 205.3%
3,avg dev 20.37,farmland: 11.6%,,,Fishing Village: 824.0,Fishing Village: 19.9%,Fishing Village: 4119.8k,Fishing Village: 136.6%
4,,forest: 10.9%,,,Forest Village: 346.4,Forest Village: 8.4%,Forest Village: 1731.8k,Forest Village: 57.4%


### Example: Top Regions in Europe

Using `run_standard_analysis` with a filter for `super_region`.

In [6]:
europe_results = analyzer.run_standard_analysis(df_locations, filters={'super_region': 'europe'})

for building, data in europe_results.items():
    print(f"\nTop European Regions for {building}:")
    display(data[['Total Bonus_mean', 'Location Count']])


Top European Regions for Fruit Orchard:


,Total Bonus_mean,Location Count
Region,,
italy_region,6.89,269
iberia_region,6.03,329
france_region,5.83,422
balkan_region,5.56,335
south_german_region,5.34,219
north_german_region,5.21,324
great_britain_region,4.78,273
caucasus_region,4.56,47
carpathia_region,4.53,127



Top European Regions for Sheep Farm:


,Total Bonus_mean,Location Count
Region,,
france_region,7.37,474
iberia_region,7.26,494
south_german_region,7.25,373
north_german_region,7.22,413
italy_region,6.99,350
great_britain_region,6.62,301
caucasus_region,5.89,207
carpathia_region,5.87,266
steppes_region,5.82,327



Top European Regions for Farming Village:


,Total Bonus_mean,Location Count
Region,,
north_german_region,8.86,419
france_region,8.84,471
south_german_region,8.59,363
italy_region,8.46,346
baltic_region,7.80,364
ruthenia_region,7.71,306
carpathia_region,7.67,261
great_britain_region,7.20,303
iberia_region,7.15,484



Top European Regions for Fishing Village:


,Total Bonus_mean,Location Count
Region,,
iberia_region,6.74,187
italy_region,6.72,188
steppes_region,6.13,37
north_german_region,6.11,222
great_britain_region,5.79,257
france_region,5.31,365
baltic_region,5.27,49
balkan_region,5.21,234
south_german_region,5.16,40



Top European Regions for Forest Village:


,Total Bonus_mean,Location Count
Region,,
south_german_region,6.61,348
ural_region,6.38,211
north_german_region,6.22,242
ruthenia_region,6.16,306
france_region,6.11,216
italy_region,5.96,189
russian_region,5.92,594
carpathia_region,5.78,264
iberia_region,5.75,262


### Example: Outlier Analysis for Scandinavia

Using `get_outlier_analysis` with a filter for `region`.

In [7]:
top, bottom = analyzer.get_outlier_analysis(df_locations, building_name="Farming Village", filters={'region': 'scandinavian_region'})

display(top[['Location', 'Province', "Development", 'Total Bonus', 'Environmental Bonus', 'Dev Bonus', 'Pop Bonus', 'Climate', 'Topography', 'Vegetation']])

,Location,Province,Development,Total Bonus,Environmental Bonus,Dev Bonus,Pop Bonus,Climate,Topography,Vegetation
103,aarhus,eastern_jutland_province,19.25,13.75,10.0,2.89,0.86,continental,flatland,farmland
3,uppsala,uppland_province,17.00,12.97,10.0,2.55,0.42,continental,flatland,farmland
53,alvastra,ostergotland_province,16.00,12.58,10.0,2.40,0.18,continental,flatland,farmland
70,falkoping,vastergotland_province,16.00,12.57,10.0,2.40,0.17,continental,flatland,farmland
113,helsingor,zealand_province,17.25,12.37,9.0,2.59,0.78,continental,flatland,grasslands
139,bregne,blekinge_province,21.00,12.30,9.0,3.15,0.15,continental,flatland,grasslands
119,navskov,funen_province,18.00,12.30,9.0,2.70,0.60,oceanic,flatland,farmland
115,kobenhavn,zealand_province,23.00,12.26,10.0,3.45,0.81,continental,flatland,farmland
102,kolding,eastern_jutland_province,18.00,12.13,9.0,2.70,0.43,oceanic,flatland,farmland
117,vordingborg,zealand_province,18.00,12.09,9.0,2.70,0.39,oceanic,flatland,farmland
